# Step 2 — Create Metadata & Audit Tables

**Purpose:** Run-once notebook that provisions the two Delta tables required by the metadata-driven pipeline:

| Table | Role |
|---|---|
| `config_ingestion` | Control table — drives every pipeline run |
| `audit_log` | Run history — one row per source per execution |

**How to run:** Attach this notebook to the `lh_ingestion_demo` Lakehouse, then click **Run all**. The notebook is idempotent — re-running it overwrites the config seed and re-creates an empty `audit_log` only if you ask it to (see the safety guard in section 4).

> ℹ️ Aligned with [`Tutorial-Metadata-Driven-Pipeline-Fabric.md`](../Tutorial-Metadata-Driven-Pipeline-Fabric.md) §5.

## 1. Sanity check — confirm the default Lakehouse is attached

In [ ]:
# Fabric notebooks expose the attached Lakehouse via the `spark` session.
# This cell fails fast if no Lakehouse is attached.
current_db = spark.catalog.currentDatabase()
print(f"Current database (Lakehouse): {current_db}")

if current_db.lower() in ("default", ""):
    raise RuntimeError(
        "No Lakehouse appears to be attached. "
        "Attach `lh_ingestion_demo` via Explorer → Add Lakehouse, then re-run."
    )

## 2. Create and seed `config_ingestion`

Schema (7 columns):

| Column | Type | Description |
|---|---|---|
| `source_id` | INT | Primary key (logical) |
| `source_system` | STRING | Friendly source name |
| `source_path` | STRING | Path in ADLS Gen2 (relative to the container) |
| `file_format` | STRING | `csv` or `parquet` |
| `target_table` | STRING | Destination Delta table |
| `load_mode` | STRING | `full` or `incremental` |
| `is_active` | BOOLEAN | `true` = include in next run |

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, BooleanType
)

config_schema = StructType([
    StructField("source_id",     IntegerType(), False),
    StructField("source_system", StringType(),  False),
    StructField("source_path",   StringType(),  False),
    StructField("file_format",   StringType(),  False),
    StructField("target_table",  StringType(),  False),
    StructField("load_mode",     StringType(),  False),
    StructField("is_active",     BooleanType(), False),
])

config_rows = [
    Row(source_id=1, source_system="adls_sales",
        source_path="raw/sales/customers.csv",
        file_format="csv", target_table="customers",
        load_mode="full", is_active=True),
    Row(source_id=2, source_system="adls_sales",
        source_path="raw/sales/orders.csv",
        file_format="csv", target_table="orders",
        load_mode="full", is_active=True),
    Row(source_id=3, source_system="adls_sales",
        source_path="raw/sales/products.csv",
        file_format="csv", target_table="products",
        load_mode="full", is_active=True),
    Row(source_id=4, source_system="adls_sales",
        source_path="raw/sales/legacy.csv",
        file_format="csv", target_table="legacy",
        load_mode="full", is_active=False),  # disabled on purpose
]

config_df = spark.createDataFrame(config_rows, schema=config_schema)

(config_df.write
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .format("delta")
          .saveAsTable("config_ingestion"))

print("✔ config_ingestion created/refreshed")

In [ ]:
# Quick preview
spark.table("config_ingestion").orderBy("source_id").show(truncate=False)

## 3. Define the `audit_log` schema

Lightweight 9-column audit table — answers *"what ran, when, how long, did it work, how many rows?"*

| Column | Type | Description |
|---|---|---|
| `run_id` | STRING | Pipeline run ID (`@pipeline().RunId`) |
| `source_id` | INT | FK to `config_ingestion.source_id` |
| `target_table` | STRING | Destination Delta table |
| `start_time` | TIMESTAMP | Per-source copy start |
| `end_time` | TIMESTAMP | Per-source copy end |
| `duration_sec` | INT | `end_time - start_time` in seconds |
| `rows_copied` | BIGINT | `rowsCopied` reported by the Copy activity |
| `status` | STRING | `success` or `failed` |
| `error_message` | STRING | Activity error text on failure, NULL on success |

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, TimestampType
)

audit_schema = StructType([
    StructField("run_id",        StringType(),    False),
    StructField("source_id",     IntegerType(),   True),
    StructField("target_table",  StringType(),    True),
    StructField("start_time",    TimestampType(), True),
    StructField("end_time",      TimestampType(), True),
    StructField("duration_sec",  IntegerType(),   True),
    StructField("rows_copied",   LongType(),      True),
    StructField("status",        StringType(),    False),
    StructField("error_message", StringType(),    True),
])

print("✔ audit_schema defined")

## 4. Create `audit_log` — safe by default

Set `RECREATE_AUDIT = True` if you want to **wipe** an existing `audit_log` (e.g. during initial setup). Leave it `False` to preserve history when re-running the notebook.

In [ ]:
RECREATE_AUDIT = False  # ⚠️ set to True only on initial setup or intentional reset

audit_exists = spark.catalog.tableExists("audit_log")

if audit_exists and not RECREATE_AUDIT:
    print("ℹ audit_log already exists — leaving it untouched.")
    print(f"  Current row count: {spark.table('audit_log').count()}")
else:
    write_mode = "overwrite" if audit_exists else "errorifexists"
    (spark.createDataFrame([], audit_schema)
          .write
          .mode(write_mode)
          .option("overwriteSchema", "true")
          .format("delta")
          .saveAsTable("audit_log"))
    action = "recreated" if audit_exists else "created"
    print(f"✔ audit_log {action} (empty)")

In [ ]:
# Confirm schema landed as expected
spark.table("audit_log").printSchema()

## 5. Final verification

Both tables should appear in the Lakehouse Explorer under **Tables/** after this cell completes.

In [ ]:
summary = spark.sql("""
    SELECT 'config_ingestion' AS table_name, COUNT(*) AS row_count FROM config_ingestion
    UNION ALL
    SELECT 'audit_log',                       COUNT(*)              FROM audit_log
""")
summary.show(truncate=False)

print("\n✅ Step 2 complete. Next: Step 3 — create the ADLS Gen2 connection.")